# Piloto 02 — avaliação da recuperação RAG

Este notebook executa um **baseline lexical reproduzível** sobre o corpus local de discursos: indexação em trechos, BM25, TF–IDF e uma combinação simples dos dois. O objetivo desta rodada é validar o encadeamento corpus → trechos → consulta → ranking → métricas.

O conjunto de consultas é **provisório**: cada consulta é derivada do resumo de um discurso e mantém o identificador desse discurso como referência conhecida. Isso permite testar o pipeline, mas não constitui ainda um conjunto-ouro anotado por especialistas nem sustenta conclusões sobre a qualidade final do sistema. A etapa seguinte deve substituir essas consultas por perguntas elaboradas e julgadas independentemente.

In [ ]:
from pathlib import Path
import re, unicodedata, hashlib, json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

ROOT = Path.cwd()
if not (ROOT / "dados").exists() and (ROOT.parent / "dados").exists():
    ROOT = ROOT.parent
DATA = ROOT / "dados/data/full/discursos_2019-02-01_2023-01-31.parquet"
OUT = ROOT / "dados/outputs"
PILOT = ROOT / "dados/piloto"
OUT.mkdir(parents=True, exist_ok=True)
PILOT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(DATA, DATA.exists())

In [ ]:
df = pd.read_parquet(DATA)
df["id"] = df["id"].astype(str)
df["Data"] = pd.to_datetime(df["Data"], errors="coerce")
df["texto"] = df["TextoDiscursoIntegral"].fillna("").astype(str)
df.loc[df["texto"].str.len() < 100, "texto"] = df.loc[df["texto"].str.len() < 100, "Resumo"].fillna("").astype(str)
df["ano"] = df["Data"].dt.year
df = df[df["texto"].str.len() >= 100].copy()
print(f"{len(df):,} documentos com texto utilizável")
print(df["ano"].value_counts().sort_index())

In [ ]:
# Conjunto provisório: um documento por ano e complementação determinística.
# A consulta usa termos do resumo apenas para validar o encadeamento técnico.
stop = set("a ao aos as com da das de do dos e em entre é este esta foi no nos o os para por que se sem sob sua suas um uma uma ao à às ou na nas pelo pela pelos pelas como sobre".split())
def make_query(summary):
    words = re.findall(r"[\wÀ-ÿ]+", str(summary).lower())
    words = [w for w in words if len(w) > 3 and w not in stop]
    return " ".join(words[:14])

usable = df[df["Resumo"].fillna("").str.len() >= 45].sort_values(["ano", "id"])
selected = usable.groupby("ano", sort=True).head(2).to_dict("records")
remaining = usable[~usable["id"].isin([r["id"] for r in selected])]
if len(selected) < 12:
    selected.extend(remaining.sample(n=12-len(selected), random_state=SEED).to_dict("records"))
refs = pd.DataFrame(selected[:12]).copy()
refs["consulta"] = refs["Resumo"].map(make_query)
refs = refs[["id", "ano", "Casa", "NomeAutor", "Resumo", "consulta"]]
refs.to_csv(PILOT / "consultas_referencia_provisorias.csv", index=False)
refs
# Quando a curadoria for concluída, crie bateria_validada.csv.
# O notebook passa a usar apenas linhas com status=validada.
validated = PILOT / "bateria_validada.csv"
if validated.exists():
    v = pd.read_csv(validated, dtype={"id_pergunta": str, "id": str})
    v = v[v["status"].eq("validada")].copy()
    if not v.empty and "pergunta" in v.columns and "id" in v.columns:
        meta = df.set_index("id")[["ano", "Casa", "NomeAutor", "Resumo"]]
        refs = v.rename(columns={"pergunta": "consulta"}).join(meta, on="id", rsuffix="_meta")
        refs = refs[["id", "ano", "Casa", "NomeAutor", "Resumo", "consulta"]]
        print(f"Usando {len(refs)} perguntas validadas.")


### Construção dos trechos

O índice usa janelas de 160 palavras com sobreposição de 32. O piloto indexa uma amostra estratificada de até 600 documentos, sempre incluindo os 12 documentos de referência. Esse limite reduz o custo da primeira execução; a avaliação final deverá usar o corpus completo ou justificar uma amostragem.

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    return re.sub(r"\s+", " ", s).strip()

# Amostra estratificada por ano, preservando documentos de referência.
ref_ids = set(refs["id"])
base = df[df["ano"].notna()].copy()
parts = []
for _, g in base.groupby("ano", sort=True):
    parts.append(g.sample(n=min(len(g), 600 // base["ano"].nunique()), random_state=SEED))
index_docs = pd.concat(parts, ignore_index=True).drop_duplicates("id")
missing = base[base["id"].isin(ref_ids) & ~base["id"].isin(set(index_docs["id"]))]
index_docs = pd.concat([index_docs, missing], ignore_index=True).drop_duplicates("id")

rows = []
WINDOW, OVERLAP = 160, 32
for row in index_docs.itertuples():
    words = norm(row.texto).split()
    step = WINDOW - OVERLAP
    for start in range(0, max(1, len(words)), step):
        chunk = words[start:start+WINDOW]
        if len(chunk) < 20: continue
        rows.append({"doc_id": row.id, "chunk_id": f"{row.id}:{start}", "text": " ".join(chunk), "ano": row.ano})
        if start + WINDOW >= len(words): break
chunks = pd.DataFrame(rows)
print(f"{len(index_docs):,} documentos → {len(chunks):,} trechos")

In [ ]:
# Índices lexical (BM25) e TF–IDF.
texts = chunks["text"].tolist()
cv = CountVectorizer(ngram_range=(1,2), min_df=2, max_features=30000)
X_count = cv.fit_transform(texts).tocsr()
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=30000, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(texts).tocsr()
terms = cv.vocabulary_
doc_len = np.asarray(X_count.sum(axis=1)).ravel()
avgdl = doc_len.mean()
k1, b = 1.5, 0.75
idf = np.log(1 + (X_count.shape[0] - np.asarray((X_count > 0).sum(axis=0)).ravel() + 0.5) / (np.asarray((X_count > 0).sum(axis=0)).ravel() + 0.5))
print(X_count.shape, X_tfidf.shape)

def bm25_scores(query):
    q = cv.transform([norm(query)])
    scores = np.zeros(X_count.shape[0], dtype=float)
    for j in q.indices:
        col = X_count[:, j].toarray().ravel()
        denom = col + k1 * (1 - b + b * doc_len / avgdl)
        scores += idf[j] * (col * (k1 + 1) / np.maximum(denom, 1e-12))
    return scores

def tfidf_scores(query):
    q = tfidf.transform([norm(query)])
    return (X_tfidf @ q.T).toarray().ravel()

def normalize(x):
    lo, hi = np.nanmin(x), np.nanmax(x)
    return (x-lo)/(hi-lo) if hi > lo else np.zeros_like(x)

def ranking(query, method):
    bm, tf = bm25_scores(query), tfidf_scores(query)
    score = bm if method == "bm25" else tf if method == "tfidf" else (normalize(bm)+normalize(tf))/2
    order = np.argsort(-score)
    return order, score

In [ ]:
def evaluate(method, k_values=(1,5,10)):
    records, details = [], []
    for qrow in refs.itertuples():
        order, scores = ranking(qrow.consulta, method)
        ranked_docs = []
        for i in order:
            d = chunks.iloc[i].doc_id
            if d not in ranked_docs: ranked_docs.append(d)
            if len(ranked_docs) >= max(k_values): break
        try: rank = ranked_docs.index(qrow.id) + 1
        except ValueError: rank = None
        details.append({"metodo": method, "id_referencia": qrow.id, "consulta": qrow.consulta, "rank_primeiro_trecho_do_documento": rank, "top10_documentos": ranked_docs[:10]})
        records.append({"metodo": method, "id_referencia": qrow.id, "rank": rank, "recall@1": int(rank is not None and rank <= 1), "recall@5": int(rank is not None and rank <= 5), "recall@10": int(rank is not None and rank <= 10), "mrr": 1/rank if rank else 0.0})
    return pd.DataFrame(records), pd.DataFrame(details)

all_metrics, all_details = [], []
for method in ["bm25", "tfidf", "hybrid"]:
    m, d = evaluate(method); all_metrics.append(m); all_details.append(d)
metrics = pd.concat(all_metrics, ignore_index=True)
summary = metrics.groupby("metodo", as_index=False)[["recall@1","recall@5","recall@10","mrr"]].mean()
summary.to_csv(OUT / "piloto_recuperacao_metricas_20260929.csv", index=False)
pd.concat(all_details, ignore_index=True).to_json(OUT / "piloto_recuperacao_detalhes_20260929.json", orient="records", force_ascii=False, indent=2)
summary

## Leitura dos resultados

As métricas acima são apenas um diagnóstico do pipeline em um conjunto de referência provisório. Elas não medem ainda a capacidade de responder perguntas reais, a suficiência documental, o grounding, a verificabilidade de citações ou a qualidade da geração. Antes de comparar arquiteturas, devemos: (1) formular perguntas independentes dos resumos; (2) associar documentos e trechos relevantes por julgamento humano; (3) ampliar o índice para o corpus completo; e (4) acrescentar um baseline neural quando o modelo e a configuração forem fixados.